# Большое домашнее задание 2

In [16]:
#pip install sacrebleu comet-ml

In [17]:
#!git clone https://github.com/k1ngofdarks/DL-Transformers.git

In [18]:
#!mv DL-Transformers DL_Transformers
#!ls

In [19]:
import torch

try:
    from data_utils import create_dataloaders, load_data_splits
    from modeling import TransformerModel
    from train_utils import train_model, translate_and_save
except:
    from DL_Transformers.data_utils import create_dataloaders, load_data_splits
    from DL_Transformers.modeling import TransformerModel
    from DL_Transformers.train_utils import train_model, translate_and_save


In [20]:
train_de, train_en, val_de, val_en, test_de, data_folder = load_data_splits()
print("Train size:", len(train_de))
print("Val   size:", len(val_de))
print("Test  size:", len(test_de))
print("DATA_FOLDER:", data_folder)


Train size: 195915
Val   size: 986
Test  size: 2998
DATA_FOLDER: bhw2-data/data


In [21]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
#print(f"AMP dtype: {amp_dtype}")


device: cuda


In [22]:
BATCH_SIZE = 128
MAX_LEN = 80


de_tokenizer, en_tokenizer, train_loader, val_loader = create_dataloaders(
    train_de=train_de,
    train_en=train_en,
    val_de=val_de,
    val_en=val_en,
    batch_size=BATCH_SIZE,
    max_len=MAX_LEN,
)


In [23]:
from comet_ml import Experiment

experiment = Experiment(
    api_key="1kc44ZRsKlVzok3SMdIgY8E6v",
    project_name="dl-transformers"
)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/alexander-stepanenko/dl-transformers/2a9aba509bbe4e849b6412a73b62e2af



In [24]:
model = TransformerModel(
    src_vocab=len(de_tokenizer),
    tgt_vocab=len(en_tokenizer),
    pad_id_src=de_tokenizer.word2id["<pad>"],
    pad_id_tgt=en_tokenizer.word2id["<pad>"],
    d_model=256,
    nhead=8,
    num_encoder_layers=4,
    num_decoder_layers=4,
    dim_feedforward=512,
    dropout=0.1,
    max_len=100,
).to(DEVICE)

train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    src_tokenizer=de_tokenizer,
    tgt_tokenizer=en_tokenizer,
    val_src_texts=val_de,
    val_tgt_texts=val_en,
    epochs=15,
    device=DEVICE,
    comet_experiment=experiment,
)


Epoch 1/15 - train:   0%|          | 0/1531 [00:00<?, ?it/s]

/home/alst/HSE/venv/lib/python3.12/site-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


OutOfMemoryError: CUDA out of memory. Tried to allocate 772.00 MiB. GPU 0 has a total capacity of 7.62 GiB of which 257.19 MiB is free. Including non-PyTorch memory, this process has 7.36 GiB memory in use. Of the allocated memory 4.69 GiB is allocated by PyTorch, and 2.54 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
translate_and_save(
    model=model,
    src_sentences=test_de,
    src_tokenizer=de_tokenizer,
    tgt_tokenizer=en_tokenizer,
    device=DEVICE,
    filename="translation_test1.en",
    batch_size=128,
    comet_experiment=None,
)
